# 🌿 CIFAR-10 Autoencoder — Full Pipeline
**Explore → Preprocess → Build → Train → Visualize Before/After Backprop**

## 1. Setup & Imports

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


## 2. Load CIFAR-10 Dataset

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),           # [0, 255] -> [0.0, 1.0]
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # [0,1] -> [-1, 1]
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True, transform=transform)
test_dataset  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=128, shuffle=False, num_workers=2)

CLASS_NAMES = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

print(f'Train samples : {len(train_dataset)}')
print(f'Test  samples : {len(test_dataset)}')

## 3. Dataset Exploration

In [ ]:
# ── Helper: denormalize for display ──────────────────────────────────────────
def denorm(tensor):
    """[-1,1] -> [0,1] for matplotlib display."""
    return tensor * 0.5 + 0.5

# ── 3a. Sample grid: one image per class ─────────────────────────────────────
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('CIFAR-10 — One Sample Per Class', fontsize=14, fontweight='bold')

shown = {}
for img, label in train_dataset:
    if label not in shown:
        shown[label] = img
    if len(shown) == 10:
        break

for idx, ax in enumerate(axes.flat):
    img = denorm(shown[idx]).permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(CLASS_NAMES[idx], fontsize=11)
    ax.axis('off')

plt.tight_layout()
plt.savefig('exploration_class_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: exploration_class_samples.png')

In [ ]:
# ── 3b. Class distribution ───────────────────────────────────────────────────
labels_all = [label for _, label in train_dataset]
counts = np.bincount(labels_all)

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(CLASS_NAMES, counts, color=plt.cm.tab10(np.linspace(0,1,10)))
ax.set_title('Class Distribution — Training Set', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of images')
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            str(count), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('exploration_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: exploration_class_distribution.png')

In [ ]:
# ── 3c. Pixel statistics per channel ─────────────────────────────────────────
batch_imgs, _ = next(iter(train_loader))   # (128, 3, 32, 32)
batch_imgs_raw = denorm(batch_imgs)        # back to [0,1]

channel_names = ['Red', 'Green', 'Blue']
channel_colors = ['red', 'green', 'blue']

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle('Pixel Value Distribution per Channel (1 batch)', fontsize=13, fontweight='bold')

for c, (name, color) in enumerate(zip(channel_names, channel_colors)):
    data = batch_imgs_raw[:, c, :, :].numpy().flatten()
    axes[c].hist(data, bins=50, color=color, alpha=0.7, edgecolor='black', linewidth=0.3)
    axes[c].set_title(f'{name} channel')
    axes[c].set_xlabel('Pixel value')
    axes[c].set_ylabel('Count')
    axes[c].axvline(data.mean(), color='black', linestyle='--', label=f'mean={data.mean():.2f}')
    axes[c].legend()

plt.tight_layout()
plt.savefig('exploration_pixel_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: exploration_pixel_distribution.png')

In [ ]:
# ── 3d. Mean image per class ──────────────────────────────────────────────────
class_sums   = {i: torch.zeros(3, 32, 32) for i in range(10)}
class_counts = {i: 0 for i in range(10)}

for img, label in train_dataset:
    class_sums[label]   += denorm(img)
    class_counts[label] += 1

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('Mean Image Per Class', fontsize=14, fontweight='bold')

for idx, ax in enumerate(axes.flat):
    mean_img = (class_sums[idx] / class_counts[idx]).permute(1, 2, 0).numpy()
    mean_img = np.clip(mean_img, 0, 1)
    ax.imshow(mean_img)
    ax.set_title(CLASS_NAMES[idx])
    ax.axis('off')

plt.tight_layout()
plt.savefig('exploration_mean_images.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: exploration_mean_images.png')

## 4. Convolutional Autoencoder Architecture

In [ ]:
class Encoder(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            # 3 x 32 x 32  ->  32 x 16 x 16
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            # 32 x 16 x 16  ->  64 x 8 x 8
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            # 64 x 8 x 8  ->  128 x 4 x 4
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, latent_dim),
        )
    def forward(self, x):
        return self.net(x)


class Decoder(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 128 * 4 * 4)
        self.net = nn.Sequential(
            # 128 x 4 x 4  ->  64 x 8 x 8
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            # 64 x 8 x 8  ->  32 x 16 x 16
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            # 32 x 16 x 16  ->  3 x 32 x 32
            nn.ConvTranspose2d(32, 3, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Tanh(),   # output in [-1, 1] to match our normalization
        )
    def forward(self, z):
        x = self.fc(z).view(-1, 128, 4, 4)
        return self.net(x)


class Autoencoder(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.encoder = Encoder(latent_dim)
        self.decoder = Decoder(latent_dim)
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)


LATENT_DIM = 128
model = Autoencoder(latent_dim=LATENT_DIM).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')
print(model)

## 5. Capture Images BEFORE Any Training (Untrained Reconstruction)

In [ ]:
# Grab a fixed test batch — we'll compare this before and after training
fixed_batch, fixed_labels = next(iter(test_loader))
fixed_batch = fixed_batch[:10].to(device)   # 10 images, one per class ideally

model.eval()
with torch.no_grad():
    recon_before = model(fixed_batch).cpu()

def show_comparison(originals, reconstructions, title, filename):
    n = len(originals)
    fig, axes = plt.subplots(2, n, figsize=(n * 1.8, 4))
    fig.suptitle(title, fontsize=13, fontweight='bold')
    axes[0, 0].set_ylabel('Original',      fontsize=10)
    axes[1, 0].set_ylabel('Reconstructed', fontsize=10)
    for i in range(n):
        orig = np.clip(denorm(originals[i]).permute(1,2,0).numpy(), 0, 1)
        rec  = np.clip(denorm(reconstructions[i]).permute(1,2,0).numpy(), 0, 1)
        axes[0, i].imshow(orig);  axes[0, i].axis('off')
        axes[1, i].imshow(rec);   axes[1, i].axis('off')
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {filename}')

show_comparison(
    fixed_batch.cpu(), recon_before,
    'BEFORE Training — Random Weights (No Backprop Yet)',
    'before_training.png'
)

## 6. Training Loop

In [ ]:
EPOCHS    = 30
LR        = 1e-3

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

train_losses = []
test_losses  = []

for epoch in range(1, EPOCHS + 1):
    # ── Train ────────────────────────────────────────────────────────────────
    model.train()
    running_loss = 0.0
    for imgs, _ in train_loader:
        imgs = imgs.to(device)
        optimizer.zero_grad()
        output = model(imgs)
        loss   = criterion(output, imgs)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)

    train_loss = running_loss / len(train_dataset)
    train_losses.append(train_loss)

    # ── Evaluate ─────────────────────────────────────────────────────────────
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for imgs, _ in test_loader:
            imgs = imgs.to(device)
            val_loss += criterion(model(imgs), imgs).item() * imgs.size(0)
    test_loss = val_loss / len(test_dataset)
    test_losses.append(test_loss)

    scheduler.step()

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch [{epoch:2d}/{EPOCHS}]  Train Loss: {train_loss:.5f}  Test Loss: {test_loss:.5f}')

print('\nTraining complete!')

## 7. Loss Curves

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(1, EPOCHS+1), train_losses, label='Train Loss', linewidth=2)
ax.plot(range(1, EPOCHS+1), test_losses,  label='Test Loss',  linewidth=2, linestyle='--')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Training & Validation Loss', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: loss_curves.png')

## 8. AFTER Training — Before vs After Backprop Side-by-Side

In [ ]:
model.eval()
with torch.no_grad():
    recon_after = model(fixed_batch).cpu()

show_comparison(
    fixed_batch.cpu(), recon_after,
    'AFTER Training — Reconstructions After Backprop',
    'after_training.png'
)

In [ ]:
# ── Full 3-row comparison: Original | Before | After ─────────────────────────
n = 10
fig, axes = plt.subplots(3, n, figsize=(n * 1.8, 6))
fig.suptitle('Original  |  Before Backprop  |  After Backprop', fontsize=13, fontweight='bold')

row_labels = ['Original', 'Before\nBackprop', 'After\nBackprop']
rows_data  = [fixed_batch.cpu(), recon_before, recon_after]

for row, (label, data) in enumerate(zip(row_labels, rows_data)):
    axes[row, 0].set_ylabel(label, fontsize=10, rotation=0, labelpad=50, va='center')
    for col in range(n):
        img = np.clip(denorm(data[col]).permute(1,2,0).numpy(), 0, 1)
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(CLASS_NAMES[fixed_labels[col].item()], fontsize=8)

plt.tight_layout()
plt.savefig('before_vs_after_backprop.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: before_vs_after_backprop.png')

## 9. Reconstruction Error Per Class

In [ ]:
model.eval()
class_losses  = {i: [] for i in range(10)}
mse_per_pixel = nn.MSELoss(reduction='none')

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs   = imgs.to(device)
        recons = model(imgs)
        losses = mse_per_pixel(recons, imgs).mean(dim=[1,2,3]).cpu().numpy()
        for loss_val, label in zip(losses, labels.numpy()):
            class_losses[label].append(loss_val)

means = [np.mean(class_losses[i]) for i in range(10)]
stds  = [np.std(class_losses[i])  for i in range(10)]

fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.bar(CLASS_NAMES, means, yerr=stds, capsize=4,
              color=plt.cm.tab10(np.linspace(0,1,10)), alpha=0.85, edgecolor='black', linewidth=0.4)
ax.set_title('Reconstruction MSE per Class (Test Set)', fontsize=13, fontweight='bold')
ax.set_ylabel('MSE Loss')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('reconstruction_error_per_class.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reconstruction_error_per_class.png')

## 10. Latent Space Visualization (t-SNE)

In [ ]:
from sklearn.manifold import TSNE

model.eval()
all_z, all_labels = [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        z = model.encoder(imgs.to(device)).cpu().numpy()
        all_z.append(z)
        all_labels.append(labels.numpy())
        if len(all_z) * 128 >= 5000:
            break  # use 5k samples to keep t-SNE fast

all_z      = np.concatenate(all_z)[:5000]
all_labels = np.concatenate(all_labels)[:5000]

print('Running t-SNE... (may take ~30s)')
tsne   = TSNE(n_components=2, random_state=42, perplexity=40, n_iter=1000)
z_2d   = tsne.fit_transform(all_z)

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(z_2d[:, 0], z_2d[:, 1],
                     c=all_labels, cmap='tab10', alpha=0.5, s=8)
cbar = plt.colorbar(scatter, ax=ax, ticks=range(10))
cbar.set_ticklabels(CLASS_NAMES)
ax.set_title('t-SNE of Latent Space (5k test samples)', fontsize=13, fontweight='bold')
ax.set_xlabel('t-SNE dim 1')
ax.set_ylabel('t-SNE dim 2')
plt.tight_layout()
plt.savefig('tsne_latent_space.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: tsne_latent_space.png')

## 11. Save the Model

In [ ]:
torch.save(model.state_dict(), 'cifar10_autoencoder.pth')
print('Model saved to cifar10_autoencoder.pth')

# To reload later:
# model = Autoencoder(latent_dim=128)
# model.load_state_dict(torch.load('cifar10_autoencoder.pth'))